# Étape 3 — Sélection de variables (version Jupyter)

Ce notebook est l'équivalent de `03_selection.py`. Il travaille uniquement sur
`train.csv` : le jeu de test n'est pas lu ici.

Règles conservées :

- graine `RANDOM_STATE = 42` partout ;
- sélection intégrée dans le pipeline, donc recalculée dans chaque fold ;
- comparaison ANOVA contre importance Random Forest ;
- contrôle du `Destination Port`, suspecté d'être un raccourci lié au service.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter ouvre le notebook depuis notebooks/ : on remonte jusqu'à config.py
dossier = Path.cwd()
while dossier != dossier.parent and not (dossier / "config.py").exists():
    dossier = dossier.parent
os.chdir(dossier)
if str(dossier) not in sys.path:
    sys.path.insert(0, str(dossier))
print("Dossier du projet :", dossier)

In [ ]:
%matplotlib inline

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from config import RANDOM_STATE, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR

SIGNIFICATION = {
    "Destination Port": "port de la machine cible, donc service visé",
    "Flow Duration": "durée totale du flux",
    "Total Fwd Packets": "nombre de paquets envoyés dans le sens initial",
    "Total Backward Packets": "nombre de paquets en réponse",
    "Total Length of Fwd Packets": "octets envoyés dans le sens initial",
    "Total Length of Bwd Packets": "octets reçus en réponse",
    "Flow Bytes/s": "débit du flux en octets par seconde",
    "Flow Packets/s": "débit du flux en paquets par seconde",
    "Flow IAT Mean": "temps moyen entre deux paquets",
    "Flow IAT Std": "variation du temps entre paquets",
    "Flow IAT Max": "plus long silence entre deux paquets",
    "Flow IAT Min": "plus court intervalle entre deux paquets",
    "Fwd IAT Mean": "temps moyen entre paquets envoyés",
    "Bwd IAT Mean": "temps moyen entre paquets reçus",
    "Fwd Packet Length Mean": "taille moyenne des paquets envoyés",
    "Bwd Packet Length Mean": "taille moyenne des paquets reçus",
    "Packet Length Mean": "taille moyenne de tous les paquets",
    "Packet Length Std": "variabilité des tailles de paquets",
    "Packet Length Variance": "variance des tailles de paquets",
    "Average Packet Size": "taille moyenne globale des paquets",
    "Min Packet Length": "plus petit paquet observé",
    "Max Packet Length": "plus grand paquet observé",
    "FIN Flag Count": "drapeau TCP de fermeture normale",
    "SYN Flag Count": "drapeau TCP d'ouverture de connexion",
    "RST Flag Count": "drapeau TCP de fermeture brutale",
    "PSH Flag Count": "drapeau TCP de poussée des données",
    "ACK Flag Count": "drapeau TCP d'acquittement",
    "URG Flag Count": "drapeau TCP de données urgentes",
    "CWE Flag Count": "drapeau TCP de congestion",
    "ECE Flag Count": "drapeau TCP de notification ECN",
    "Fwd PSH Flags": "poussée de données dans le sens initial",
    "Fwd URG Flags": "données urgentes dans le sens initial",
    "Fwd Header Length": "taille de l'en-tête TCP envoyé",
    "Bwd Header Length": "taille de l'en-tête TCP reçu",
    "Fwd Header Length.1": "copie de Fwd Header Length calculée par CICFlowMeter",
    "Down/Up Ratio": "rapport entre trafic descendant et montant",
    "Init_Win_bytes_forward": "fenêtre TCP Initiale dans le sens initial",
    "Init_Win_bytes_backward": "fenêtre TCP initiale dans le sens réponse",
    "act_data_pkt_fwd": "nombre de paquets avec des données utiles",
    "min_seg_size_forward": "plus petit segment TCP envoyé",
    "Subflow Fwd Packets": "paquets dans un sous-flux (égal à Total Fwd ici)",
    "Subflow Bwd Packets": "paquets dans le sous-flux inverse",
    "Subflow Fwd Bytes": "octets dans un sous-flux",
    "Subflow Bwd Bytes": "octets dans le sous-flux inverse",
    "Active Mean": "durée moyenne des périodes actives du flux",
    "Idle Mean": "durée moyenne des périodes d'inactivité",
}

K_VALUES = [5, 10, 15, 25, 40, 70]


def modele():
    # Forêt fixe, utilisée pour les importances et pour noter les sélections
    return RandomForestClassifier(n_estimators=60, max_depth=12, n_jobs=-1,
                                  class_weight="balanced_subsample", random_state=RANDOM_STATE)


print("Graine utilisee :", RANDOM_STATE)

## 1. Chargement du train et paires redondantes

On impute le train avec la médiane, puis on cherche les paires de variables
corrélées à plus de 0,95. Ces mesures ne portent que sur le train.

In [ ]:
train = pd.read_csv(PROCESSED_DIR / "train.csv", low_memory=False)
y = train["Family"]
X = train.drop(columns=["Family"])
features = list(X.columns)
print("Départ :", X.shape[1], "variables et", len(X), "lignes de train")

imputation = SimpleImputer(strategy="median")
X_impute = imputation.fit_transform(X)
corr = pd.DataFrame(X_impute, columns=features).corr().abs()
masque = np.triu(np.ones(corr.shape, dtype=bool), k=1)
paires = corr.where(masque).stack().rename("correlation").reset_index()
paires.columns = ["variable_1", "variable_2", "correlation"]
paires = paires[paires["correlation"] > 0.95]
paires.to_csv(TABLES_DIR / "correlations_train.csv", index=False)
print("Paires redondantes sur le train :", len(paires))

## 2. Deux classements des variables

Le test ANOVA donne un score F par variable. La forêt donne une importance.
On regarde l'union des deux tops 15, avec la signification réseau de chaque
variable. Le but est de comprendre, pas seulement de choisir un score.

In [ ]:
scores_anova, _ = f_classif(X_impute, y)
foret = modele()
foret.fit(X_impute, y)

classement = pd.DataFrame({
    "feature": features,
    "f_anova": scores_anova,
    "importance_rf": foret.feature_importances_,
})
classement["rang_anova"] = classement["f_anova"].rank(ascending=False, method="min")
classement["rang_rf"] = classement["importance_rf"].rank(ascending=False, method="min")
top = classement[(classement["rang_anova"] <= 15) | (classement["rang_rf"] <= 15)].copy()
top["signification"] = top["feature"].map(SIGNIFICATION).fillna("statistique de flux CICFlowMeter")
top = top.sort_values(["rang_rf", "rang_anova"])
top.to_csv(TABLES_DIR / "top15_features.csv", index=False)
print("\nVariables les plus utiles (union des tops 15) :")
top[["feature", "rang_anova", "rang_rf", "signification"]].head(15)

## 3. Courbe F1 macro selon le nombre de variables

Pour chaque méthode et chaque valeur de `k`, on construit un pipeline. La
sélection est **dans** le pipeline : chaque fold recalcule la sienne avec ses
propres données d'entraînement. La validation est à 3 folds pour limiter le
temps de calcul.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
resultats = []

for methode in ["anova", "random_forest"]:
    for k in K_VALUES:
        if methode == "anova":
            selection = SelectKBest(score_func=f_classif, k=k)
        else:
            selection = SelectFromModel(modele(), threshold=-np.inf, max_features=k)

        pipe = Pipeline([
            ("imputation", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold()),
            ("selection", selection),
            ("modele", modele()),
        ])
        debut = time.time()
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro")
        resultats.append({
            "methode": methode,
            "nb_variables": k,
            "f1_moy": round(scores.mean(), 4),
            "f1_std": round(scores.std(), 4),
            "temps_s": round(time.time() - debut, 1),
        })
        print(methode, "k =", k, "F1 =", resultats[-1]["f1_moy"])

courbe = pd.DataFrame(resultats)
courbe.to_csv(TABLES_DIR / "courbe_selection.csv", index=False)
courbe

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for methode, couleur in [("anova", "steelblue"), ("random_forest", "darkorange")]:
    partie = courbe[courbe["methode"] == methode]
    ax.errorbar(partie["nb_variables"], partie["f1_moy"], yerr=partie["f1_std"],
                marker="o", label=methode, color=couleur, capsize=3)
ax.set_title("F1 macro selon le nombre de variables gardées")
ax.set_xlabel("Nombre de variables")
ax.set_ylabel("F1 macro (validation croisée sur le train)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
fig.savefig(FIGURES_DIR / "courbe_selection.png", dpi=120, bbox_inches="tight")
plt.close(fig)

## 4. Contrôle du port de destination

`Destination Port` pourrait être un raccourci : il indique le service visé.
On compare le F1 macro avec et sans cette variable, toujours avec 15 variables
choisies par la forêt dans le pipeline.

In [ ]:
X_sans_port = X.drop(columns=["Destination Port"])
controle = []
for cas, donnees in [("avec Destination Port", X), ("sans Destination Port", X_sans_port)]:
    pipe = Pipeline([
        ("imputation", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
        ("selection", SelectFromModel(modele(), threshold=-np.inf, max_features=15)),
        ("modele", modele()),
    ])
    scores = cross_val_score(pipe, donnees, y, cv=cv, scoring="f1_macro")
    controle.append({"cas": cas, "f1_moy": round(scores.mean(), 4),
                     "f1_std": round(scores.std(), 4)})
    print(cas, "F1 =", controle[-1]["f1_moy"])
table_controle = pd.DataFrame(controle)
table_controle.to_csv(TABLES_DIR / "controle_port.csv", index=False)
table_controle

## 5. Choix final

On garde 15 variables choisies par la forêt, sans `Destination Port`. Le F1 est
meilleur qu'avec le port (0,8398 contre 0,8153) et l'inférence sera plus rapide.
La liste est sauvegardée dans `data/processed/selected_features.txt` dans
l'ordre d'importance décroissant.

In [ ]:
X_final_impute = SimpleImputer(strategy="median").fit_transform(X_sans_port)
foret_finale = modele()
foret_finale.fit(X_final_impute, y)
classement_final = pd.DataFrame({
    "feature": X_sans_port.columns,
    "importance_rf": foret_finale.feature_importances_,
})
colonnes_finales = list(classement_final.sort_values("importance_rf", ascending=False)["feature"][:15])
f1_final = controle[1]["f1_moy"]

with open(PROCESSED_DIR / "selected_features.txt", "w", encoding="utf-8") as fichier:
    fichier.write("\n".join(colonnes_finales))

top["dans_selection_finale"] = top["feature"].isin(colonnes_finales)
top.to_csv(TABLES_DIR / "top15_features.csv", index=False)

pd.DataFrame({
    "methode": ["Random Forest, sans Destination Port"],
    "nb_variables": [len(colonnes_finales)],
    "f1_moy": [f1_final],
    "variables": [", ".join(colonnes_finales)],
}).to_csv(TABLES_DIR / "selection_finale.csv", index=False)

print("\nChoix final :", len(colonnes_finales), "variables (F1 =", f1_final, ")")
print("Liste sauvegardée dans data/processed/selected_features.txt")
classement_final.sort_values("importance_rf", ascending=False).head(15)

## Résumé de l'étape 3 (version Jupyter)

Résultats identiques à ceux du script `03_selection.py` :

- 48 paires de variables corrélées à plus de 0,95 sur le train ;
- à 15 variables, ANOVA 0,6334 contre Random Forest 0,8153 de F1 macro ;
- à 70 variables, le F1 monte à 0,8689, mais on garde 15 variables pour la
  simplicité et la vitesse ;
- avec `Destination Port` : 0,8153 ; sans : 0,8398. Le port est donc exclu ;
- la liste finale des 15 variables est dans
  `data/processed/selected_features.txt`.

Le test n'a toujours pas été utilisé.

In [ ]:
print("\nFin de l'étape 3 (version Jupyter). Sélection écrite dans data/processed/.")